# TCP Calibration for the myCobot 280 Labs
## Lab 5: Flange-to-TCP Estimation, Validation, and Workflow Integration

Run this notebook from `~/venvs/mycobot`:

```bash
source ~/venvs/mycobot/bin/activate
jupyter lab
```

This lab continues Labs 1–4. You will work with real measurement data from the physical myCobot 280 to estimate the rigid geometric offset between the robot flange and the actual tool point (TCP). The estimated TCP is then validated and used in later Cartesian robot workflows.

All positions in this notebook are converted to **metres** and orientations to **radians** for computation. The raw measurement data is recorded in millimetres and degrees, matching the myCobot API convention.

In [ ]:
%matplotlib widget
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import roboticstoolbox as rtb

from spatialmath import SE3
from helperFunctions import (
    rpy_to_rot, rpy_to_transform, rot_to_vec, transform_to_pipi,
)

np.set_printoptions(precision=6)

## Setup

### 1. Import the myCobot 280 URDF model

Load the URDF model distributed alongside this notebook. The kinematic chain runs from `g_base` to `joint6_flange`. The flange frame is the last robot frame before any tool correction — the TCP frame is what we are calibrating.

In [ ]:
notebook_path = os.getcwd()
mycobot_urdf_path = os.path.join(notebook_path, 'mycobot_280_gazebo.urdf')
mycobot = rtb.Robot.URDF(mycobot_urdf_path)
print(mycobot)

for link in mycobot.links:
    print(link.name)

### 2. Identify the relevant frames

The two frames central to this lab are:

- **Base frame:** `g_base` — the fixed robot base used by the URDF and the controller.
- **Flange frame:** `joint6_flange` — the last link in the kinematic chain before any tool is attached.

The **TCP frame** is rigidly attached to the mounted tool. Its offset from the flange is the unknown we will estimate. After calibration, the offset is applied through the launch parameters `ik_tool_xyz` and `ik_tool_rpy` so that IK and Cartesian commands refer to the calibrated TCP rather than the raw flange.

In [ ]:
base_link = 'g_base'
flange_link = 'joint6_flange'

# Verify FK at home configuration
q_home = np.zeros(mycobot.n)
T_home = mycobot.fkine(q_home, end=flange_link)
print('Flange pose at home:\n', T_home)

## Measurement Data and Sample Structure

### 3. Load the calibration samples

Each calibration sample consists of three measurements recorded on the physical myCobot 280:

- **Joint angles** in degrees (read from the robot encoders)
- **Measured TCP position** in millimetres (from the myCobot API `get_coords`)
- **Measured TCP orientation** as roll-pitch-yaw in degrees (from the same API)

The samples below are the actual measurements used in the codebase calibration script `fk_tool_calib.py`. Each row represents one robot pose where the tool point was recorded.

In [ ]:
# Actual calibration samples from fk_tool_calib.py
# Format: (joint_angles_deg, tcp_xyz_mm, tcp_rpy_deg)
samples = [
    ([-0.87, -2.02, -89.91, 89.38, 1.05, 0.7],
     [147.7, -64.8, 319.4],
     [-92.54, 0.74, -89.85]),

    ([-0.52, -0.87, -0.61, -0.08, 0.96, 0.61],
     [51.1, -63.1, 419.5],
     [-91.58, 0.64, -89.66]),

    ([9.49, 20.56, 29.61, 40.86, 49.3, 59.41],
     [-180.7, -59.4, 332.6],
     [-44.61, -23.67, -11.74]),

    ([59.23, 51.15, 39.9, 31.02, 20.83, 10.63],
     [-96.6, -254.5, 203.1],
     [30.49, -23.21, -32.58]),

    ([-59.15, -50.71, -40.34, 39.28, 49.3, 59.41],
     [107.9, -236.9, 230.6],
     [158.88, 56.69, 158.02]),
]

print(f'Number of calibration samples: {len(samples)}')

### 4. Convert one sample to SI units

Convert the first sample from the raw measurement units to the computation units used in this notebook:
- Joint angles: degrees → radians
- TCP position: millimetres → metres
- TCP orientation: degrees → radians

In [ ]:
angles_deg, tcp_xyz_mm, tcp_rpy_deg = samples[0]

q_rad = ...  # Convert the encoder joint angles to radians
tcp_xyz_m = ...  # Convert the measured TCP position to metres
tcp_rpy_rad = ...  # Convert the measured TCP orientation to radians

print('q (rad):', q_rad)
print('TCP xyz (m):', tcp_xyz_m)
print('TCP rpy (rad):', tcp_rpy_rad)

## Per-Sample Flange-to-TCP Transform

### 5. Compute the flange-to-TCP transform for one sample

For each measurement sample, the flange-to-TCP transform is obtained by:

1. Computing the flange pose ${}^B\mathbf{T}_{F,i}$ from FK using the recorded joint angles.
2. Assembling the measured TCP pose ${}^B\mathbf{T}_{\text{TCP},i}$ from the recorded position and RPY orientation using the codebase RPY convention (`rpy_to_transform`).
3. Computing the per-sample offset: ${}^F\mathbf{T}_{T,i} = \left({}^B\mathbf{T}_{F,i}\right)^{-1} {}^B\mathbf{T}_{\text{TCP},i}$

In [ ]:
# Step 1: FK to get flange pose in base frame
T_base_flange = ...  # Compute the flange pose in the base frame

# Step 2: Assemble measured TCP pose using codebase RPY convention
T_base_tcp = ...  # Build the measured TCP pose from position and orientation

# Step 3: Per-sample flange-to-TCP transform
T_flange_tcp = ...  # Compute the relative transform from flange to TCP

print('Flange pose (base frame):\n', T_base_flange)
print('Measured TCP pose (base frame):\n', T_base_tcp)
print('Flange-to-TCP transform:\n', T_flange_tcp)
print('TCP offset (mm):', T_flange_tcp[:3, 3] * 1000.0)

### 6. Compute flange-to-TCP transforms for all samples

Repeat the per-sample computation over all calibration samples. Store the resulting transforms, translation vectors, and rotation matrices separately.

In [ ]:
flange_to_tcp_transforms = []
flange_to_tcp_translations = []
flange_to_tcp_rotations = []

for angles_deg, tcp_xyz_mm, tcp_rpy_deg in samples:
    q_rad = ...  # convert to radians
    tcp_xyz_m = ...  # convert to metres
    tcp_rpy_rad = ...  # convert to radians

    T_base_flange = ...  # Compute the flange pose in the base frame
    T_base_tcp = ...  # Build the measured TCP pose from position and orientation
    T_flange_tcp = ...  # Compute the relative transform from flange to TCP

    flange_to_tcp_transforms.append(T_flange_tcp)
    flange_to_tcp_translations.append(T_flange_tcp[:3, 3])
    flange_to_tcp_rotations.append(T_flange_tcp[:3, :3])

print(f'Computed {len(flange_to_tcp_transforms)} per-sample transforms.')
for i, t in enumerate(flange_to_tcp_translations):
    print(f'  Sample {i+1} offset (mm): {t * 1000.0}')

## Estimate a Common TCP Transform

### 7. Average the translational offset

Average the per-sample TCP translation vectors component-wise to obtain a single estimated translation offset.

In [ ]:
mean_translation = ...  # Compute the average TCP translation across all samples

print('Mean TCP offset (m):', mean_translation)
print('Mean TCP offset (mm):', mean_translation * 1000.0)

### 8. SVD-based mean rotation

Average the per-sample rotation matrices using an SVD projection. Simply averaging rotation matrices element-wise does not produce a valid rotation. The SVD-based approach:

1. Sum all rotation matrices.
2. Compute the SVD of the sum.
3. Project back to SO(3): $\bar{\mathbf{R}} = \mathbf{U}\mathbf{V}^T$.
4. Check the determinant: if $\det(\bar{\mathbf{R}}) < 0$, negate the last column of $\mathbf{U}$ and recompute.

This matches the `mean_rotation` function in the codebase's `fk_tool_calib.py`.

In [ ]:
def mean_rotation_svd(rotations):
    """SVD-based mean rotation (matches fk_tool_calib.py)."""
    R_sum = ...  # sum all rotation matrices
    U, _, Vt = np.linalg.svd(R_sum)
    R_mean = ...  # Construct the mean rotation from the SVD result
    # Ensure proper rotation (det > 0)
    if np.linalg.det(R_mean) < 0:
        ...  # Correct the result if the rotation is improper
    return R_mean

R_mean = mean_rotation_svd(flange_to_tcp_rotations)
print('Mean rotation:\n', R_mean)
print('det(R_mean):', np.linalg.det(R_mean))

### 9. Assemble the estimated flange-to-TCP transform

Combine the averaged translation and the SVD-mean rotation into a single 4×4 homogeneous transform. This is the calibrated flange-to-TCP offset.

In [ ]:
T_flange_tcp_est = np.eye(4)
T_flange_tcp_est[:3, :3] = ...  # Insert the estimated mean rotation
T_flange_tcp_est[:3, 3] = ...  # Insert the estimated mean translation

print('Estimated flange-to-TCP transform:\n', T_flange_tcp_est)
print('Estimated TCP offset (mm):', T_flange_tcp_est[:3, 3] * 1000.0)

## Validate the Calibration

### 10. Reprojection: reconstruct TCP positions from the estimated offset

For each sample, reconstruct the TCP position using the estimated flange-to-TCP transform:

$${}^B\mathbf{T}_{\text{TCP,est},i} = {}^B\mathbf{T}_{F,i} \cdot {}^F\hat{\mathbf{T}}_T$$

Compare the reconstructed TCP position with the originally measured TCP position.

In [ ]:
reconstructed_positions = []
measured_positions = []

for i, (angles_deg, tcp_xyz_mm, tcp_rpy_deg) in enumerate(samples):
    q_rad = np.radians(angles_deg)
    tcp_xyz_m = np.array(tcp_xyz_mm) / 1000.0

    T_base_flange = ...  # Compute the flange pose in the base frame
    T_base_tcp_est = ...  # Reconstruct the TCP pose from the flange pose and calibration

    reconstructed_positions.append(T_base_tcp_est[:3, 3])
    measured_positions.append(tcp_xyz_m)

reconstructed_positions = np.array(reconstructed_positions)
measured_positions = np.array(measured_positions)

for i in range(len(samples)):
    print(f'Sample {i+1}: measured={measured_positions[i]*1000} mm, '
          f'reconstructed={reconstructed_positions[i]*1000} mm')

### 11. Compute residual error measures

Compute the residual (difference between reconstructed and measured TCP positions) for each sample. Report the per-sample error norm, the mean residual norm, the maximum residual, and the RMS position error.

In [ ]:
residuals = ...  # Compute the position residuals
residual_norms = ...  # Compute one residual magnitude per sample
mean_residual = ...
max_residual = ...
rms_error = ...  # Compute the RMS position error

print('Per-sample residual norms (mm):')
for i, rn in enumerate(residual_norms):
    print(f'  Sample {i+1}: {rn * 1000.0:.4f} mm')
print(f'Mean residual: {mean_residual * 1000.0:.4f} mm')
print(f'Max residual:  {max_residual * 1000.0:.4f} mm')
print(f'RMS error:     {rms_error * 1000.0:.4f} mm')

### 12. Plot residual norms

Visualise the residual norm for each measurement sample. A consistent calibration should show small, evenly distributed residuals. One unusually large residual may indicate a measurement outlier.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(samples) + 1), residual_norms * 1000.0)
ax.set_xlabel('Sample index')
ax.set_ylabel('Residual norm (mm)')
ax.set_title('TCP calibration residual per sample')
ax.grid(True, axis='y')
plt.tight_layout()

### 13. Interpret the calibration quality

Inspect your residual measures and answer briefly:
- Are the residuals small relative to the myCobot 280's expected repeatability (~0.5–1 mm)?
- Is any single sample an obvious outlier?
- Would removing an outlier and re-estimating improve the calibration?

In [ ]:
# Space for student interpretation — no code required, but students may
# optionally re-run the estimation with a subset of samples here.

## Compare with Codebase Calibration Result

### 14. Compare against the launch file calibration values

The codebase launch file (`mycobot_control.launch.py`) contains the calibrated tool offset that was estimated from the same measurement workflow:

```python
'ik_tool_xyz': [-0.0008809655, -0.0017207283, 0.0001719996],
'ik_tool_rpy': [4.91e-05, 2.52e-04, 5.37e-05],
```

Compare your estimated offset with these values. Small differences may come from numerical precision or a slightly different averaging approach.

In [ ]:
# Known calibration result from the launch file
launch_tool_xyz = np.array([-0.0008809655, -0.0017207283, 0.0001719996])
launch_tool_rpy = np.array([4.91e-05, 2.52e-04, 5.37e-05])

# Your estimated translation
est_xyz = T_flange_tcp_est[:3, 3]

print('Launch ik_tool_xyz (mm):', launch_tool_xyz * 1000.0)
print('Your estimate (mm):     ', est_xyz * 1000.0)
print('Difference (mm):        ', (est_xyz - launch_tool_xyz) * 1000.0)

### 15. Base correction transform (optional exploration)

The codebase calibration script (`fk_tool_calib.py`) also estimates a base correction transform that compensates for a residual mismatch between the nominal URDF base frame and the frame in which the measurements were taken. This correction is applied through the launch parameters `ik_base_xyz` and `ik_base_rpy`.

This is an advanced optional step. If you want to explore base correction, inspect the `fk_tool_calib.py` script to see how it computes and averages per-sample base corrections alongside the tool offset. The key idea is that any systematic offset between the URDF base and the real measurement frame can be absorbed into a small corrective transform.

In [ ]:
# Optional: base correction estimation
# base_corrections = []
# for i, (angles_deg, tcp_xyz_mm, tcp_rpy_deg) in enumerate(samples):
#     ...
#     base_corr_i = ...  # Compute the per-sample base correction transform
#     base_corrections.append(base_corr_i)
# ...

## Extract Launch Parameters

### 16. Extract `ik_tool_xyz` and `ik_tool_rpy`

Extract the translation and RPY orientation from the estimated flange-to-TCP transform. These become the `ik_tool_xyz` and `ik_tool_rpy` parameters in the controller launch file.

To convert a rotation matrix back to `[roll, pitch, yaw]`, use the inverse of the RPY-to-rotation formula from the codebase. The convention is the same as in `helperFunctions.py`: `R = Rz(yaw) @ Ry(pitch) @ Rx(roll)`.

In [ ]:
def rot_to_rpy(rot):
    """
    Convert rotation matrix to [roll, pitch, yaw].
    Inverse of rpy_to_rot: R = Rz(yaw) @ Ry(pitch) @ Rx(roll).
    Matches rot_to_rpy in fk_tool_calib.py.
    """
    sy = -rot[2, 0]
    sy = max(-1.0, min(1.0, sy))
    pitch = math.asin(sy)
    cp = math.cos(pitch)
    if abs(cp) < 1e-6:
        roll = 0.0
        yaw = math.atan2(-rot[0, 1], rot[1, 1])
    else:
        roll = math.atan2(rot[2, 1] / cp, rot[2, 2] / cp)
        yaw = math.atan2(rot[1, 0] / cp, rot[0, 0] / cp)
    return [roll, pitch, yaw]


ik_tool_xyz = ...  # Extract the calibrated tool translation
ik_tool_rpy = ...  # Convert the calibrated tool rotation to RPY

print('ik_tool_xyz:', ik_tool_xyz.tolist())
print('ik_tool_rpy:', ik_tool_rpy)

### 17. How the calibrated TCP is used in the controller

The controller applies the tool offset in the IK solver (`_solve_ik` in `mycobot_control.py`):

1. At startup, the tool transform is built from `ik_tool_xyz` and `ik_tool_rpy`.
2. Its **inverse** is precomputed: `_ik_tool_t_inv = inv(T_tool)`.
3. When a Cartesian target arrives on `/mycobot/ee_pose`, the target is adjusted: `target_t = target_t @ _ik_tool_t_inv`.

This means Cartesian commands are interpreted at the **calibrated TCP** rather than at the raw flange. A correct TCP offset is therefore essential for accurate Cartesian positioning with the myCobot 280.

## Reflection

### 18. Short reflection questions

Answer briefly based on your calibration experience:

1. Why does an incorrect TCP create systematic Cartesian positioning error even when joint angles are correct?
2. Why are several sufficiently different measurement poses better than one or two?
3. Why must the calibrated TCP be validated (e.g., with reprojection error) before use in later robot workflows?

In [ ]:
# Space for student answers (text in comments or markdown).